# Single-track train delay propagation

How much waiting does a late train cause at a zero-buffer crossing? The synthetic timetable represents Peradeniya Junction, Gampola and Nawalapitiya. Historical scheduled journey times are used as fixed segment durations.

In [1]:
import csv
from collections import defaultdict, deque
from heapq import heappop, heappush
from io import StringIO
from pathlib import Path

# Embedded copy lets this small example run in a fresh Colab session.
embedded_csv = """train,origin,destination,departure,arrival
A,Peradeniya Junction,Gampola,09:39,10:00
A,Gampola,Nawalapitiya,10:02,10:27
B,Nawalapitiya,Gampola,09:31,09:58
B,Gampola,Peradeniya Junction,10:00,10:18
C,Peradeniya Junction,Gampola,10:18,10:39
C,Gampola,Nawalapitiya,10:41,11:06
"""
paths = [Path("data/baseline_timetable.csv"), Path("../data/baseline_timetable.csv")]
source = next((path for path in paths if path.is_file()), None)
rows = list(csv.DictReader(StringIO(source.read_text() if source else embedded_csv)))

def minute(clock):
    hour, minutes = map(int, clock.split(":"))
    return 60 * hour + minutes

def clock(minutes):
    return f"{minutes // 60:02d}:{minutes % 60:02d}"

for row in rows:
    row["start"] = minute(row["departure"])
    row["duration"] = minute(row["arrival"]) - row["start"]
    row["segment"] = tuple(sorted((row["origin"], row["destination"])))

## Movement rules

Each segment admits one train at a time in scheduled departure order. A train can depart once its timetable permits, its entry delay or two-minute station dwell has elapsed, and the segment is free. The clock jumps to the next readiness or arrival event. All arrivals at the same minute release their segments before departures. For this three-service plan the fixed segment orders allow at most two trains at Gampola.

In [2]:
def simulate(entry_delays, timetable=None):
    if timetable is None:
        timetable = rows
    orders = defaultdict(deque)
    routes = defaultdict(list)
    for i in sorted(range(len(timetable)), key=lambda i: timetable[i]["start"]):
        orders[timetable[i]["segment"]].append(i)
        routes[timetable[i]["train"]].append(i)

    events = []
    next_leg = {}
    for train, legs in routes.items():
        first = legs[0]
        heappush(events, (timetable[first]["start"] + entry_delays.get(train, 0), "ready", first))
        next_leg.update(zip(legs, legs[1:]))

    occupied = set()
    ready_since = {}
    movements = {}
    while events:
        now = events[0][0]
        while events and events[0][0] == now:
            _, event, i = heappop(events)
            if event == "ready":
                ready_since[i] = now
            else:
                occupied.remove(timetable[i]["segment"])
                if i in next_leg:
                    j = next_leg[i]
                    heappush(events, (max(now + 2, timetable[j]["start"]), "ready", j))

        for segment, order in orders.items():
            if segment in occupied or not order or order[0] not in ready_since:
                continue
            i = order.popleft()
            arrival = now + timetable[i]["duration"]
            movements[i] = (now, arrival, now - ready_since.pop(i))
            occupied.add(segment)
            heappush(events, (arrival, "arrive", i))

    assert len(movements) == len(timetable), "Some trains could not finish"
    return movements

## Check against the hand calculation

The third value recorded for each movement is the time spent waiting after the train was ready and its timetable permitted departure. An imposed entry delay is counted separately.

In [3]:
baseline = simulate({})
disrupted = simulate({"A": 25})
for i, row in enumerate(rows):
    assert baseline[i] == (row["start"], minute(row["arrival"]), 0)
    depart, arrive, wait = disrupted[i]
    print(f"{row['train']}  {row['origin']} to {row['destination']}: "
          f"{clock(depart)} to {clock(arrive)}, conflict wait {wait} min")

other_wait = sum(disrupted[i][2] for i, row in enumerate(rows) if row["train"] != "A")
assert other_wait == 50
for train, origin, expected in [("B", "Gampola", "10:25"), ("C", "Peradeniya Junction", "10:43")]:
    i = next(i for i, r in enumerate(rows) if (r["train"], r["origin"]) == (train, origin))
    assert disrupted[i][0] == minute(expected) and disrupted[i][2] == 25
print(f"Additional waiting across B and C: {other_wait} train-minutes.")

A  Peradeniya Junction to Gampola: 10:04 to 10:25, conflict wait 0 min
A  Gampola to Nawalapitiya: 10:27 to 10:52, conflict wait 0 min
B  Nawalapitiya to Gampola: 09:31 to 09:58, conflict wait 0 min
B  Gampola to Peradeniya Junction: 10:25 to 10:43, conflict wait 25 min
C  Peradeniya Junction to Gampola: 10:43 to 11:04, conflict wait 25 min
C  Gampola to Nawalapitiya: 11:06 to 11:31, conflict wait 0 min
Additional waiting across B and C: 50 train-minutes.


## Five-minute crossing buffer at Gampola

Schedule B to leave Gampola five minutes later and shift both legs of C by five minutes so C can use the segment after B. B's scheduled journey increases by five minutes. C starts and finishes five minutes later with the same scheduled journey duration. Compare on-time and disrupted runs against this revised timetable.

In [4]:
def crossing_timetable(buffer_minutes):
    timetable = []
    for original in rows:
        leg = original.copy()
        if leg["train"] == "C" or (leg["train"] == "B" and leg["origin"] == "Gampola"):
            leg["start"] += buffer_minutes
            leg["departure"] = clock(leg["start"])
            leg["arrival"] = clock(leg["start"] + leg["duration"])
        timetable.append(leg)
    return timetable

buffered = crossing_timetable(5)
on_time = simulate({}, buffered)
late = simulate({"A": 25}, buffered)
for i, leg in enumerate(buffered):
    assert on_time[i] == (leg["start"], minute(leg["arrival"]), 0)

for train, station, scheduled, actual in [
    ("B", "Gampola", "10:05", "10:25"),
    ("C", "Peradeniya Junction", "10:23", "10:43"),
]:
    i = next(i for i, leg in enumerate(buffered)
             if leg["train"] == train and leg["origin"] == station)
    assert on_time[i][0] == minute(scheduled)
    assert late[i][0] == minute(actual) and late[i][2] == 20
    print(f"{train} at {station}: scheduled {scheduled}, departed {actual}, extra wait {late[i][2]} min")

extra_wait = sum(late[i][2] - on_time[i][2] for i, leg in enumerate(buffered) if leg["train"] != "A")
assert extra_wait == 40
print(f"Additional waiting across B and C: {extra_wait} train-minutes")

B at Gampola: scheduled 10:05, departed 10:25, extra wait 20 min
C at Peradeniya Junction: scheduled 10:23, departed 10:43, extra wait 20 min
Additional waiting across B and C: 40 train-minutes


## Compare four crossing buffers

Only A enters late, by 25 minutes. For each buffer, compare a disrupted run with an on-time run on the same timetable. Report the extra conflict waiting for B and C in train-minutes. The timetable cost is the increase in minutes from the first scheduled departure to the last scheduled arrival.

In [5]:
base_span = max(minute(r["arrival"]) for r in rows) - min(r["start"] for r in rows)
print("Buffer (min) | B wait | C wait | Total (train-min) | Added timetable span (min)")
for buffer in (0, 5, 10, 15):
    schedule = crossing_timetable(buffer)
    normal = simulate({}, schedule)
    disrupted_run = simulate({"A": 25}, schedule)
    waits = defaultdict(int)
    for i, leg in enumerate(schedule):
        assert normal[i] == (leg["start"], minute(leg["arrival"]), 0)
        waits[leg["train"]] += disrupted_run[i][2] - normal[i][2]
    total = waits["B"] + waits["C"]
    span = max(minute(r["arrival"]) for r in schedule) - min(r["start"] for r in schedule)
    assert waits["B"] == waits["C"] == max(0, 25 - buffer)
    assert span - base_span == buffer
    print(f"{buffer:12} | {waits['B']:6} | {waits['C']:6} | {total:17} | {span - base_span:26}")

Buffer (min) | B wait | C wait | Total (train-min) | Added timetable span (min)
           0 |     25 |     25 |                50 |                          0
           5 |     20 |     20 |                40 |                          5
          10 |     15 |     15 |                30 |                         10
          15 |     10 |     10 |                20 |                         15


## Random entry delays

Each run samples independent entry delays for A, B and C from an exponential distribution with an assumed mean of five minutes. The draws retain fractional minutes. This distribution is a modelling assumption. The fixed running times and crossing order still apply.

Generate the three delays before dispatch and reuse them for every buffer. For each timetable, first run with those delays alone. Then add 25 minutes to A's sampled delay and run again. Subtract the first run's total conflict waiting for B and C from the second run's total. This measures the extra waiting attributable to the disruption.

Repeat with 1,000 independent sets of draws. The approximate 95% interval for each mean is the sample mean plus or minus 1.96 sample standard errors. It measures uncertainty from the finite number of runs under these assumptions.

In [6]:
from random import Random
from statistics import mean, stdev
from math import sqrt

SEED = 42
MEAN_ENTRY_MINUTES = 5
REPLICATIONS = 1000
train_ids = sorted({leg["train"] for leg in rows})
rng = Random(SEED)
entry_samples = [
    {train: rng.expovariate(1 / MEAN_ENTRY_MINUTES) for train in train_ids}
    for _ in range(REPLICATIONS)
]
print("First run, background entry delays (minutes):")
for train, delay in entry_samples[0].items():
    print(f"{train}: {delay:.2f}")

First run, background entry delays (minutes):
A: 5.10
B: 0.13
C: 1.61


In [7]:
def paired_extra_wait(schedule, background):
    disrupted_entries = dict(background)
    disrupted_entries["A"] += 25
    normal = simulate(background, schedule)
    disrupted = simulate(disrupted_entries, schedule)
    return sum(disrupted[i][2] - normal[i][2]
               for i, leg in enumerate(schedule) if leg["train"] != "A")

print("Buffer (min) | Mean additional B+C wait (train-min) | Approx. 95% CI")
for buffer in (0, 5, 10, 15):
    schedule = crossing_timetable(buffer)
    # Switching off background lateness must recover the hand calculation.
    assert paired_extra_wait(schedule, dict.fromkeys(train_ids, 0)) == 2 * (25 - buffer)
    effects = [paired_extra_wait(schedule, sample) for sample in entry_samples]
    # With this fixed order each of B and C can inherit at most 25 extra minutes.
    assert all(-1e-9 <= value <= 50 + 1e-9 for value in effects)
    average = mean(effects)
    margin = 1.96 * stdev(effects) / sqrt(len(effects))
    print(f"{buffer:12} | {average:36.2f} | {average-margin:.2f} to {average+margin:.2f}")

Buffer (min) | Mean additional B+C wait (train-min) | Approx. 95% CI
           0 |                                43.62 | 43.07 to 44.18
           5 |                                40.40 | 39.84 to 40.97
          10 |                                33.68 | 33.08 to 34.27
          15 |                                25.23 | 24.61 to 25.84
